In [4]:
pip install numpy pandas matplotlib nltk scikit-learn scipy

#  SECTION 1: IMPORTS AND SETUP

In [5]:
import numpy as np
import pandas as pd
import re
import string
import nltk
import os

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Setup complete.")


Setup complete.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


# Section 2: Dataset Loading


In [6]:
from google.colab import drive
drive.mount('/content/drive')

# ── FOLDER STRUCTURE ─────────────────────────────────────────────────────────
# Recommended layout inside your Google Drive:
#
#   MyDrive/
#   └── Reuters21578/
#       ├── datasets/          ← place all CSV files here
#       ├── checkpoints/       ← pickle files (wsd_docs.pkl, etc.)
#       ├── chain_outputs/     ← lexical chain inspection outputs
#       └── logs/              ← optional run logs
#
# Update DATASET_DIR below to match wherever you placed the CSV files.

DATASET_DIR   = "/content/drive/MyDrive/WSD_LexicalChains/Reuters-21578/datasets/"
CHECKPOINT_DIR = "/content/drive/MyDrive/WSD_LexicalChains/Reuters-21578/checkpoints/"
CHAIN_OUT_DIR  = "/content/drive/MyDrive/WSD_LexicalChains/Reuters-21578/chain_outputs/"

# Create output directories if they don't already exist
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(CHAIN_OUT_DIR,  exist_ok=True)

# ── CHECKPOINT PATH ───────────────────────────────────────────────────────────
WSD_CACHE_FILE = os.path.join(CHECKPOINT_DIR, 'wsd_docs.pkl')

print("Drive mounted and paths configured.")
print(f"  Dataset dir   : {DATASET_DIR}")
print(f"  Checkpoint dir: {CHECKPOINT_DIR}")
print(f"  Chain out dir : {CHAIN_OUT_DIR}")
print(f"  WSD cache     : {WSD_CACHE_FILE}")


Mounted at /content/drive
Drive mounted and paths configured.
  Dataset dir   : /content/drive/MyDrive/WSD_LexicalChains/Reuters-21578/datasets/
  Checkpoint dir: /content/drive/MyDrive/WSD_LexicalChains/Reuters-21578/checkpoints/
  Chain out dir : /content/drive/MyDrive/WSD_LexicalChains/Reuters-21578/chain_outputs/
  WSD cache     : /content/drive/MyDrive/WSD_LexicalChains/Reuters-21578/checkpoints/wsd_docs.pkl


In [7]:
# Reuters-21578 comes in three standard splits:
#
#   ModApte  — most widely used in research; train/test boundary is cleanest
#              (only documents explicitly assigned to train OR test are kept).
#              ~9,600 training + ~3,700 test documents.
#
#   ModLewis — retains more borderline documents; slightly larger but noisier.
#
#   ModHayes — preserves even more documents; rarely used in modern work.


# we're using ModApte

In [8]:
import ast

TRAIN_FILE = os.path.join(DATASET_DIR, 'ModApte_train.csv')
TEST_FILE  = os.path.join(DATASET_DIR, 'ModApte_test.csv')

for fpath in [TRAIN_FILE, TEST_FILE]:
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  ✓  Found: {os.path.basename(fpath)}  ({size_mb:.2f} MB)")
    else:
        print(f"  ✗  MISSING: {fpath}")
        print("     → Check that DATASET_DIR is set correctly above.")

print("\nLoading Reuters-21578 ModApte split...")
df_train = pd.read_csv(TRAIN_FILE)
df_test  = pd.read_csv(TEST_FILE)
df_all   = pd.concat([df_train, df_test], ignore_index=True)
print(f"  Train rows : {len(df_train)}")
print(f"  Test rows  : {len(df_test)}")
print(f"  Combined   : {len(df_all)}")

print(f"\nColumns: {list(df_all.columns)}")

def parse_topics(val):
    if isinstance(val, list):
        return val
    if not isinstance(val, str) or val.strip() in ('', '[]', 'nan'):
        return []
    try:
        parsed = ast.literal_eval(val)
        return parsed if isinstance(parsed, list) else []
    except Exception:
        return []

df_all['topics_list'] = df_all['topics'].apply(parse_topics)

df_all['title'] = df_all['title'].fillna('')
df_all['text']  = df_all['text'].fillna('')
df_all['full_text'] = (df_all['title'] + ' ' + df_all['text']).str.strip()

# drop empty docs
df_all = df_all[df_all['full_text'].str.len() > 20].copy()
print(f"\nDocuments with non-empty text: {len(df_all)}")

df_filtered = df_all[df_all['topics_list'].apply(len) > 0].copy()
print(f"Documents with at least one topic: {len(df_filtered)}")

documents = df_filtered['full_text'].tolist()
# Store topic lists alongside for reference (not used in WSD/chains)
doc_topics = df_filtered['topics_list'].tolist()

print(f"\n{'─'*50}")
print(f"FINAL CORPUS SUMMARY")
print(f"{'─'*50}")
print(f"Total documents : {len(documents)}")
print(f"\nSample document (first 500 chars):")
print(documents[0][:500])
print(f"\nCorresponding topics: {doc_topics[0]}")

  ✓  Found: ModApte_train.csv  (8.65 MB)
  ✓  Found: ModApte_test.csv  (2.80 MB)

Loading Reuters-21578 ModApte split...
  Train rows : 9603
  Test rows  : 3299
  Combined   : 12902

Columns: ['text', 'text_type', 'topics', 'lewis_split', 'cgis_split', 'old_id', 'new_id', 'places', 'people', 'orgs', 'exchanges', 'date', 'title']

Documents with non-empty text: 12902
Documents with at least one topic: 10794

──────────────────────────────────────────────────
FINAL CORPUS SUMMARY
──────────────────────────────────────────────────
Total documents : 10794

Sample document (first 500 chars):
BAHIA COCOA REVIEW Showers continued throughout the week in
the Bahia cocoa zone, alleviating the drought since early
January and improving prospects for the coming temporao,
although normal humidity levels have not been restored,
Comissaria Smith said in its weekly review.
    The dry period means the temporao will be late this year.
    Arrivals for the week ended February 22 were 155,221 bags
of 60 k

In [9]:
import ast

TRAIN_FILE = os.path.join(DATASET_DIR, 'ModApte_train.csv')
TEST_FILE  = os.path.join(DATASET_DIR, 'ModApte_test.csv')

for fpath in [TRAIN_FILE, TEST_FILE]:
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  ✓  Found: {os.path.basename(fpath)}  ({size_mb:.2f} MB)")
    else:
        print(f"  ✗  MISSING: {fpath}")
        print("     → Check that DATASET_DIR is set correctly above.")

print("\nLoading Reuters-21578 ModApte split...")
df_train = pd.read_csv(TRAIN_FILE)
df_test  = pd.read_csv(TEST_FILE)
df_all   = pd.concat([df_train, df_test], ignore_index=True)
print(f"  Train rows : {len(df_train)}")
print(f"  Test rows  : {len(df_test)}")
print(f"  Combined   : {len(df_all)}")

print(f"\nColumns: {list(df_all.columns)}")

# The 'topics' column may be stored as a string representation of a list,
# e.g.  "['earn', 'acq']"  — safely parse it into an actual Python list.
def parse_topics(val):
    if isinstance(val, list):
        return val
    if not isinstance(val, str) or val.strip() in ('', '[]', 'nan'):
        return []
    try:
        parsed = ast.literal_eval(val)
        return parsed if isinstance(parsed, list) else []
    except Exception:
        return []

df_all['topics_list'] = df_all['topics'].apply(parse_topics)

df_all['title'] = df_all['title'].fillna('')
df_all['text']  = df_all['text'].fillna('')
df_all['full_text'] = df_all['title'] + ' ' + df_all['text']
df_all['full_text']  = df_all['full_text'].str.strip()

df_all = df_all[df_all['full_text'].str.len() > 20].copy()
print(f"\nDocuments with non-empty text: {len(df_all)}")

from collections import Counter
topic_counter = Counter(t for topics in df_all['topics_list'] for t in topics)
print("\nTop 50 topics by document frequency:")
for topic, count in topic_counter.most_common(50):
    print(f"  {topic:<20} {count}")

TOP_N = 50
CHOSEN_TOPICS = [topic for topic, count in topic_counter.most_common(TOP_N)]
print(f"\nChosen topics: {CHOSEN_TOPICS}")

def single_label_category(topics):
    """Return the single matching category, or None if 0 or >1 match."""
    matches = [t for t in topics if t in CHOSEN_TOPICS]
    return matches[0] if len(matches) == 1 else None

df_all['category'] = df_all['topics_list'].apply(single_label_category)
df_filtered = df_all[df_all['category'].notna()].copy()
print(f"\nDocuments surviving single-label filter: {len(df_filtered)}")
print("Per-category counts:")
print(df_filtered['category'].value_counts())

MAX_PER_CLASS = 100   # WSD is expensiveee!!

sampled_dfs = []
for topic in CHOSEN_TOPICS:
    subset = df_filtered[df_filtered['category'] == topic]
    n_take = min(MAX_PER_CLASS, len(subset))
    sampled_dfs.append(subset.sample(n=n_take, random_state=RANDOM_STATE))

df_sample = pd.concat(sampled_dfs, ignore_index=True).sample(
    frac=1, random_state=RANDOM_STATE
).reset_index(drop=True)

TOPIC_TO_ID   = {topic: i for i, topic in enumerate(CHOSEN_TOPICS)}
documents     = df_sample['full_text'].tolist()
true_labels   = [TOPIC_TO_ID[cat] for cat in df_sample['category']]

print(f"\n{'─'*50}")
print(f"FINAL CORPUS SUMMARY")
print(f"{'─'*50}")
print(f"Topics selected    : {CHOSEN_TOPICS}")
print(f"\nSample document (first 300 chars):")
print(documents[0][:600])
print(f"\nCorresponding label: {CHOSEN_TOPICS[true_labels[0]]}")


  ✓  Found: ModApte_train.csv  (8.65 MB)
  ✓  Found: ModApte_test.csv  (2.80 MB)

Loading Reuters-21578 ModApte split...
  Train rows : 9603
  Test rows  : 3299
  Combined   : 12902

Columns: ['text', 'text_type', 'topics', 'lewis_split', 'cgis_split', 'old_id', 'new_id', 'places', 'people', 'orgs', 'exchanges', 'date', 'title']

Documents with non-empty text: 12902

Top 50 topics by document frequency:
  earn                 3923
  acq                  2292
  crude                374
  trade                326
  money-fx             293
  interest             271
  money-supply         151
  ship                 144
  money-fxinterest     137
  grainwheat           134
  sugar                122
  coffee               112
  gold                 90
  graincorn            86
  money-fxdlr          80
  gnp                  73
  cpi                  71
  cocoa                61
  grain                51
  alum                 50
  reserves             49
  jobs                 49
  crude

In [10]:
print("Number of selected topics:", len(CHOSEN_TOPICS))
print("Documents retained:", len(df_filtered))

Number of selected topics: 50
Documents retained: 9719


# section 3: Preprocessing



In [11]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# nltk data files already downloaded in Section 1

# initialise tools we will be reusing across all steps
stemmer        = PorterStemmer()
STOPWORDS      = set(stopwords.words('english'))  # common English words to remove

# After stemming, words become unrecognisable (e.g. "running" → "run").
# WordNet needs real English words, not stems !!
# So we keep a global dictionary that maps each stem back to the most common original word — used in Steps 4 and 5.
stem_to_word = {}   # e.g.  {"run": "running",  "space": "space"}

# -----------------------------------------------------------------------------
def clean_text(text):
    """
    Remove noise from a raw document string.
      1. Lowercase everything
      2. Remove URLs
      3. Remove email addresses
      4. Remove non-alphabetic characters (numbers, punctuation, etc.)
      5. Collapse multiple spaces into one
    """
    text = text.lower()                            # 1. lowercase
    text = re.sub(r'http\S+|www\.\S+', ' ', text) # 2. remove URLs
    text = re.sub(r'\S+@\S+', ' ', text)          # 3. remove emails
    text = re.sub(r'[^a-z\s]', ' ', text)         # 4. keep only a-z and spaces
    text = re.sub(r'\s+', ' ', text).strip()       # 5. collapse whitespace
    return text


# -----------------------------------------------------------------------------
def tokenize_and_filter(text):
    """
      1. Tokenize
      2. Remove stopwords
      3. Remove very short words (len < 3)
      4. Remove very long words (len > 20)
    """

    tokens = word_tokenize(text)                   # 1. split into words
    tokens = [t for t in tokens
              if t not in STOPWORDS                # 2. drop stopwords
              and 3 <= len(t) <= 20]               # 3 & 4. length filter
    return tokens


# -----------------------------------------------------------------------------
def stem_tokens(tokens):

    # Reduce each token to its root (stem)
    # updates the global stem_to_word mapping so we can recover real words for WordNet lookups later.

    stemmed = []
    for word in tokens:
        stem = stemmer.stem(word)        # e.g. "running" → "run"
        stemmed.append(stem)

        # Stores the shortest real-word form for each stem
        # (shorter originals tend to be the base form)
        if stem not in stem_to_word:
            stem_to_word[stem] = word
        else:
            # prefer the shorter word as the representative
            if len(word) < len(stem_to_word[stem]):
                stem_to_word[stem] = word

    return stemmed

# only need unstemmed tokens!
def preprocess_document(text):
    text   = clean_text(text)
    tokens = tokenize_and_filter(text)
    return tokens


# -----------------------------------------------------------------------------
def preprocess_corpus(documents):
    """
    Apply preprocess_document to every document in the corpus.
    """
    preprocessed = []
    total = len(documents)

    for i, doc in enumerate(documents):
        # Progress update every 100 documents
        if (i + 1) % 100 == 0 or (i + 1) == total:
            print(f"  Preprocessing document {i+1}/{total}...")
        preprocessed.append(preprocess_document(doc))

    return preprocessed


# preprocessing on the full corpus
print("Starting preprocessing...\n")
preprocessed_docs = preprocess_corpus(documents)
print("\nPreprocessing complete.")

print(f"\nTotal documents preprocessed : {len(preprocessed_docs)}")
print(f"Stem-to-word mapping size    : {len(stem_to_word)} stems")
print(f"\nExample — original (first 200 chars):")
print(documents[0][:200])
print(f"\nExample — preprocessed tokens (first 20):")
print(preprocessed_docs[0][:20])

Starting preprocessing...

  Preprocessing document 100/2640...
  Preprocessing document 200/2640...
  Preprocessing document 300/2640...
  Preprocessing document 400/2640...
  Preprocessing document 500/2640...
  Preprocessing document 600/2640...
  Preprocessing document 700/2640...
  Preprocessing document 800/2640...
  Preprocessing document 900/2640...
  Preprocessing document 1000/2640...
  Preprocessing document 1100/2640...
  Preprocessing document 1200/2640...
  Preprocessing document 1300/2640...
  Preprocessing document 1400/2640...
  Preprocessing document 1500/2640...
  Preprocessing document 1600/2640...
  Preprocessing document 1700/2640...
  Preprocessing document 1800/2640...
  Preprocessing document 1900/2640...
  Preprocessing document 2000/2640...
  Preprocessing document 2100/2640...
  Preprocessing document 2200/2640...
  Preprocessing document 2300/2640...
  Preprocessing document 2400/2640...
  Preprocessing document 2500/2640...
  Preprocessing document 2600/26

# SECTION 4: NOUN EXTRACTION FUNCTIONS

In [12]:
from nltk import pos_tag

# Noun POS tags
NOUN_TAGS = {'NN', 'NNS', 'NNP', 'NNPS'}


def extract_nouns_from_doc(token_list):
    tagged = pos_tag(token_list)

    nouns = [
        word
        for word, tag in tagged
        if tag in NOUN_TAGS
    ]

    return nouns


def extract_nouns_from_corpus(preprocessed_docs):
    """
    Apply noun extraction to every document.
    """
    noun_docs = []
    total = len(preprocessed_docs)

    for i, token_list in enumerate(preprocessed_docs):
        if (i + 1) % 100 == 0 or (i + 1) == total:
            print(f"  Extracting nouns from document {i+1}/{total}...")

        nouns = extract_nouns_from_doc(token_list)
        noun_docs.append(nouns)

    return noun_docs


print("Starting noun extraction...\n")

noun_docs = extract_nouns_from_corpus(preprocessed_docs)

print("\nNoun extraction complete.")

print(f"\nTotal documents processed : {len(noun_docs)}")

total_tokens = sum(len(doc) for doc in preprocessed_docs)
total_nouns  = sum(len(doc) for doc in noun_docs)

print(f"Total tokens             : {total_tokens}")
print(f"Total nouns kept         : {total_nouns}")
print(f"Noun retention rate      : {100 * total_nouns / total_tokens:.1f}%")

print(f"\nExample — tokens (first 20):")
print(preprocessed_docs[0][:20])

print(f"\nExample — nouns (first 20):")
print(noun_docs[0][:20])

Starting noun extraction...

  Extracting nouns from document 100/2640...
  Extracting nouns from document 200/2640...
  Extracting nouns from document 300/2640...
  Extracting nouns from document 400/2640...
  Extracting nouns from document 500/2640...
  Extracting nouns from document 600/2640...
  Extracting nouns from document 700/2640...
  Extracting nouns from document 800/2640...
  Extracting nouns from document 900/2640...
  Extracting nouns from document 1000/2640...
  Extracting nouns from document 1100/2640...
  Extracting nouns from document 1200/2640...
  Extracting nouns from document 1300/2640...
  Extracting nouns from document 1400/2640...
  Extracting nouns from document 1500/2640...
  Extracting nouns from document 1600/2640...
  Extracting nouns from document 1700/2640...
  Extracting nouns from document 1800/2640...
  Extracting nouns from document 1900/2640...
  Extracting nouns from document 2000/2640...
  Extracting nouns from document 2100/2640...
  Extracting n

# SECTION 5: WORD SENSE DISAMBIGUATION (WSD)


In [13]:
import os
import pickle

from nltk.corpus import wordnet as wn

try:
    from tqdm import tqdm
    TQDM_AVAILABLE = True
except ImportError:
    TQDM_AVAILABLE = False

MAX_SENSES = 5

_synset_cache = {}
_similarity_cache = {}


def get_top_senses(word):
    if word not in _synset_cache:
        synsets = wn.synsets(word, pos=wn.NOUN)
        _synset_cache[word] = synsets[:MAX_SENSES]
    return _synset_cache[word]


def wu_palmer_similarity(synset_a, synset_b):
    key = frozenset([synset_a.name(), synset_b.name()])

    if key not in _similarity_cache:
        score = synset_a.wup_similarity(synset_b)
        _similarity_cache[key] = score if score is not None else 0.0

    return _similarity_cache[key]


def disambiguate_word(target_word, context_words_deduped):
    target_senses = get_top_senses(target_word)

    if not target_senses:
        return None

    if len(target_senses) == 1:
        return target_senses[0]

    best_sense = None
    best_score = -1.0

    for candidate in target_senses:
        total_score = 0.0

        for ctx_word in context_words_deduped:
            ctx_senses = get_top_senses(ctx_word)

            if not ctx_senses:
                continue

            max_sim = max(
                wu_palmer_similarity(candidate, ctx_sense)
                for ctx_sense in ctx_senses
            )

            total_score += max_sim

        if total_score > best_score:
            best_score = total_score
            best_sense = candidate

    return best_sense


def disambiguate_document(nouns):
    all_words_deduped = list(dict.fromkeys(nouns))

    result = []

    for word in nouns:
        context = [w for w in all_words_deduped if w != word]

        best_sense = disambiguate_word(word, context)

        result.append((word, best_sense))

    return result


def disambiguate_corpus(noun_docs):
    wsd_results = []
    total = len(noun_docs)

    iterator = (
        tqdm(noun_docs, desc="WSD Progress", unit="doc")
        if TQDM_AVAILABLE
        else noun_docs
    )

    for i, nouns in enumerate(iterator):
        if not TQDM_AVAILABLE and ((i + 1) % 50 == 0 or (i + 1) == total):
            print(
                f"  WSD: document {i+1}/{total}  "
                f"| synset cache: {len(_synset_cache)} words  "
                f"| sim cache: {len(_similarity_cache)} pairs"
            )

        wsd_results.append(
            disambiguate_document(nouns)
        )

    return wsd_results


if os.path.exists(WSD_CACHE_FILE):
    print("Loading cached WSD results...")

    with open(WSD_CACHE_FILE, 'rb') as f:
        wsd_docs_names = pickle.load(f)

    wsd_docs = [
        [
            (
                word,
                wn.synset(name) if name is not None else None
            )
            for (word, name) in doc
        ]
        for doc in wsd_docs_names
    ]

    print("WSD complete.")

else:
    print("Running WSD...")
    print("(First run takes several minutes — subsequent runs load instantly)\n")

    wsd_docs = disambiguate_corpus(noun_docs)

    wsd_docs_names = [
        [
            (
                word,
                syn.name() if syn is not None else None
            )
            for (word, syn) in doc
        ]
        for doc in wsd_docs
    ]

    print("\nSaving WSD results...")

    with open(WSD_CACHE_FILE, 'wb') as f:
        pickle.dump(wsd_docs_names, f)

    print("WSD complete.")


total_nouns = sum(len(doc) for doc in wsd_docs)

matched_nouns = sum(
    1
    for doc in wsd_docs
    for (_, syn) in doc
    if syn is not None
)

print(f"\nTotal noun tokens  : {total_nouns}")
print(f"Matched to WordNet : {matched_nouns}")
print(f"WordNet coverage   : {100 * matched_nouns / total_nouns:.1f}%")
print(f"Synset cache size  : {len(_synset_cache)} unique words")
print(f"Sim cache size     : {len(_similarity_cache)} unique pairs")

print("\nExample — WSD output for first document (first 5 words):")

for word, syn in wsd_docs[0][:5]:
    if syn:
        print(
            f"  word='{word:<15}' "
            f"synset='{syn.name():<22}' "
            f"def: '{syn.definition()[:55]}...'"
        )
    else:
        print(
            f"  word='{word:<15}' "
            f"NOT FOUND in WordNet"
        )

Loading cached WSD results...
WSD complete.

Total noun tokens  : 129082
Matched to WordNet : 114847
WordNet coverage   : 89.0%
Synset cache size  : 0 unique words
Sim cache size     : 0 unique pairs

Example — WSD output for first document (first 5 words):
  word='mobil          ' NOT FOUND in WordNet
  word='mob            ' synset='syndicate.n.01        ' def: 'a loose affiliation of gangsters in charge of organized...'
  word='capital        ' synset='capital.n.01          ' def: 'assets available for use in the production of further a...'
  word='mobil          ' NOT FOUND in WordNet
  word='corp           ' synset='corporation.n.01      ' def: 'a business firm whose articles of incorporation have be...'


In [14]:

print("\nExample — WSD output for first document (first 20 words):")

for word, syn in wsd_docs[0][:20]:
    if syn:
        print(
            f"  word='{word:<15}' "
            f"synset='{syn.name():<22}' "
            f"def: '{syn.definition()[:55]}...'"
        )
    else:
        print(
            f"  word='{word:<15}' "
            f"NOT FOUND in WordNet"
        )


Example — WSD output for first document (first 20 words):
  word='mobil          ' NOT FOUND in WordNet
  word='mob            ' synset='syndicate.n.01        ' def: 'a loose affiliation of gangsters in charge of organized...'
  word='capital        ' synset='capital.n.01          ' def: 'assets available for use in the production of further a...'
  word='mobil          ' NOT FOUND in WordNet
  word='corp           ' synset='corporation.n.01      ' def: 'a business firm whose articles of incorporation have be...'
  word='chairman       ' synset='president.n.04        ' def: 'the officer who presides at the meetings of an organiza...'
  word='allen          ' synset='allen.n.02            ' def: 'United States filmmaker and comic actor (1935-)...'
  word='murray         ' synset='murray.n.02           ' def: 'Scottish philologist and the lexicographer who shaped t...'
  word='report         ' synset='report.n.02           ' def: 'the act of informing by verbal report...'
  word='today 

# SECTION 6: LEXICAL CHAIN FUNCTIONS

In [15]:
# ============================================================
# SECTION 6: LEXICAL CHAIN CONSTRUCTION  (conservative rebuild)
# ============================================================
#
# Paper: "A Semantic Approach for Text Clustering Using WordNet
#         and Lexical Chains" (2015)
#
# Key fixes over the previous version
# ------------------------------------
# 1.  NO shared-ancestor merging.
#     Old code: `if ancestors_a & ancestors_b: union(i, j)`
#     Problem : any two nouns whose WordNet chains meet within 3
#               hops — e.g. "bond" and "energy" via
#               physical_phenomenon.n.01 — were merged, even when
#               semantically unrelated in the document.
#     Fix     : only a DIRECT hypernym/hyponym relation (one synset
#               is an ancestor of the other) triggers a merge.
#
# 2.  Hypernym depth reduced from 3 → 2.
#     At depth 3, nearly every concrete noun reaches a handful of
#     WordNet super-categories that are shared by huge swaths of
#     the vocabulary.  Depth 2 keeps domain-level generalisations
#     (bank.n.02 → depository_institution.n.01 → institution.n.01)
#     without climbing to entity.n.01.
#
# 3.  Meronym depth reduced to 1 (direct only).
#     Transitive meronym traversal drags in distant parts-of-parts
#     that have no visible relationship in the text.  Direct
#     part/substance/member meronyms and their holonym inverses
#     are sufficient and faithful to the paper.
#
# 4.  Wu-Palmer fallback threshold raised from 0.5 → 0.85.
#     WP is a path-distance proxy that is notoriously liberal.
#     0.5 fires on "profit + rate" (wup≈0.50), "trade + market"
#     (wup≈0.62), creating spurious merges.  0.85 restricts it to
#     near-synonym pairs (e.g. "trade + deal" wup≈0.82) where
#     structural checks already cover most cases anyway.
#
# 5.  Wu-Palmer fallback weight reduced: treated as MEDIUM-STRONG
#     (below synonym, below direct hypernym).
#
# 6.  Linear scan replaces all-pairs Union-Find.
#     Union-Find is transitive: if A↔B and B↔C, then A+B+C merge
#     even if A and C are unrelated.  The paper's algorithm scans
#     words left-to-right and assigns each to the BEST-scoring
#     existing chain (or opens a new one).  This breaks the
#     transitivity cascade.
#
# 7.  Best-chain selection (highest aggregate weight) instead of
#     first-compatible-chain.  This places each word in the most
#     semantically coherent chain available.
#
# 8.  Minimum chain size pruning: singleton chains (size < 2) are
#     retained as raw singletons but flagged; callers can filter
#     them for downstream use.
#
# Relation weights (same labels as paper)
# -----------------------------------------
#   EXTRA_STRONG  identity / same synset                   1.0
#   STRONG        synonym (shared lemma name)              0.9
#   MEDIUM_STRONG direct hypernym/hyponym (depth ≤ 2)      0.7
#   MEDIUM        direct meronym/holonym (depth = 1)       0.4
#   WEAK          WP similarity ≥ 0.85                     0.2
# ============================================================

from nltk.corpus import wordnet as wn

# ── weights ──────────────────────────────────────────────────
WEIGHT_IDENTITY       = 1.0   # same synset
WEIGHT_SYNONYM        = 0.9   # shared lemma
WEIGHT_HYPERNYM       = 0.7   # direct hypernym / hyponym (depth ≤ 2)
WEIGHT_MERONYM        = 0.4   # direct meronym / holonym (depth = 1)
WEIGHT_WP             = 0.2   # Wu-Palmer fallback

# ── hypernym depth cap ────────────────────────────────────────
# 2 is the sweet spot: captures domain-level generalisations
# (bank → depository_institution → institution) without
# climbing to entity.n.01 / physical_entity.n.01 super-nodes.
HYPERNYM_DEPTH        = 2

# ── Wu-Palmer threshold ───────────────────────────────────────
# 0.85 allows near-synonym pairs (trade+deal ≈ 0.82) while
# blocking loosely related terms (profit+loss ≈ 0.50,
# trade+market ≈ 0.62).
WP_THRESHOLD          = 0.85

# ── minimum chain size to keep in final output ────────────────
# Set to 1 to keep singletons; raise to 2 to suppress them.
MIN_CHAIN_SIZE        = 1

# ── caches shared across documents ───────────────────────────
_hypernym_cache  = {}   # synset.name() → frozenset of ancestor synset names
_wp_cache        = {}   # frozenset({name_a, name_b}) → float


# ─────────────────────────────────────────────────────────────
def _ancestors(synset, depth=HYPERNYM_DEPTH):
    """
    Return frozenset of synset NAMES reachable upward within
    `depth` hypernym hops.  Cached per (synset, depth) pair.
    """
    key = (synset.name(), depth)
    if key not in _hypernym_cache:
        visited  = set()
        frontier = [synset]
        for _ in range(depth):
            nxt = []
            for s in frontier:
                for h in s.hypernyms():
                    if h.name() not in visited:
                        visited.add(h.name())
                        nxt.append(h)
            frontier = nxt
        _hypernym_cache[key] = frozenset(visited)
    return _hypernym_cache[key]


# ─────────────────────────────────────────────────────────────
def _direct_meronyms_holonyms(synset):
    """
    Return set of synset NAMES that are direct (depth-1)
    meronyms or holonyms of `synset`.
    """
    related = (
        synset.part_meronyms()       +
        synset.substance_meronyms()  +
        synset.member_meronyms()     +
        synset.part_holonyms()       +
        synset.substance_holonyms()  +
        synset.member_holonyms()
    )
    return frozenset(s.name() for s in related)


# ─────────────────────────────────────────────────────────────
def _wp(synset_a, synset_b):
    """Wu-Palmer similarity, cached."""
    key = frozenset([synset_a.name(), synset_b.name()])
    if key not in _wp_cache:
        score = synset_a.wup_similarity(synset_b)
        _wp_cache[key] = score if score is not None else 0.0
    return _wp_cache[key]


# ─────────────────────────────────────────────────────────────
def get_relation_weight(syn_a, syn_b):
    """
    Return the numeric relation weight between two Synset objects,
    or 0.0 if no valid relation exists.

    Relation hierarchy (checked in priority order):
      1. Identity          — same synset
      2. Synonym           — shared lemma name
      3. Direct hypernym   — one is a direct ancestor of the other
                             within HYPERNYM_DEPTH hops (no shared
                             ancestor — that caused chain explosion)
      4. Direct meronym    — direct part/substance/member or holonym
                             relation (depth = 1 only)
      5. WP fallback       — Wu-Palmer ≥ WP_THRESHOLD

    What is intentionally NOT checked:
      • Shared common ancestor  → was the main explosion source
      • Transitive meronyms     → depth > 1 meronyms cause same problem
      • WP < WP_THRESHOLD       → too liberal, causes false merges
    """
    # ① identity
    if syn_a.name() == syn_b.name():
        return WEIGHT_IDENTITY

    # ② synonym — shared lemma name
    lemmas_a = frozenset(l.name() for l in syn_a.lemmas())
    lemmas_b = frozenset(l.name() for l in syn_b.lemmas())
    if lemmas_a & lemmas_b:
        return WEIGHT_SYNONYM

    # ③ direct hypernym / hyponym (no shared-ancestor check)
    anc_a = _ancestors(syn_a, HYPERNYM_DEPTH)
    anc_b = _ancestors(syn_b, HYPERNYM_DEPTH)
    if syn_b.name() in anc_a or syn_a.name() in anc_b:
        return WEIGHT_HYPERNYM
    # ← removed:  if anc_a & anc_b  (shared ancestor — was explosion source)

    # ④ direct meronym / holonym (depth = 1)
    mh_a = _direct_meronyms_holonyms(syn_a)
    mh_b = _direct_meronyms_holonyms(syn_b)
    if syn_b.name() in mh_a or syn_a.name() in mh_b:
        return WEIGHT_MERONYM

    # ⑤ Wu-Palmer fallback (tight threshold)
    wp = _wp(syn_a, syn_b)
    if wp >= WP_THRESHOLD:
        return WEIGHT_WP

    return 0.0


# ─────────────────────────────────────────────────────────────
def _score_against_chain(syn, chain_synsets):
    """
    Score how well `syn` fits into a chain by summing relation
    weights against every synset already in the chain.

    Using the sum (rather than max) rewards chains that contain
    multiple related concepts, making dominant-topic chains
    more attractive to new words.
    """
    total = 0.0
    for cs in chain_synsets:
        total += get_relation_weight(syn, cs)
    return total


# ─────────────────────────────────────────────────────────────
def build_lexical_chains(wsd_pairs):
    """
    Build lexical chains for one document using a left-to-right
    linear scan — faithful to the paper's algorithm.

    Algorithm
    ---------
    For each (word, synset) pair in document order:
      1. Skip if synset is None (WordNet miss).
      2. Compute a compatibility score against every existing chain
         (sum of relation weights to all chain members).
      3. If the best score > 0, append to that chain.
         If multiple chains score equally, pick the largest one
         (rewards coherent growing chains).
      4. Otherwise open a new singleton chain.
    After scan, update term-frequency counters per chain.

    Why linear scan instead of Union-Find
    --------------------------------------
    Union-Find is symmetric and transitive: if A relates to B and
    B relates to C, then A,B,C all merge even if A and C share no
    direct relation.  This is the primary source of giant chains.
    Linear scan assigns each word to one chain at insertion time;
    later words in the document do not retroactively pull together
    concepts that were inserted into separate chains earlier.

    Parameters
    ----------
    wsd_pairs : list of (word_str, Synset | None)
        Output of Section 5 WSD for one document.

    Returns
    -------
    list of list of (word_str, Synset, int)
        Each inner list is one lexical chain.
        The int is the within-document term frequency of that synset.
    """
    valid = [(w, s) for (w, s) in wsd_pairs if s is not None]
    if not valid:
        return []

    # ── term-frequency counting (per synset name) ────────────
    tf = {}
    for _, syn in valid:
        tf[syn.name()] = tf.get(syn.name(), 0) + 1

    # ── deduplicate: keep first occurrence of each synset ────
    seen_names = set()
    unique = []
    for word, syn in valid:
        if syn.name() not in seen_names:
            seen_names.add(syn.name())
            unique.append((word, syn))

    # ── linear scan ──────────────────────────────────────────
    # chains_syns  : list of lists of Synset   (for fast scoring)
    # chains_words : list of lists of (word, Synset) (for output)
    chains_syns  = []
    chains_words = []

    for word, syn in unique:
        best_score  = 0.0
        best_idx    = -1

        for idx, chain_s in enumerate(chains_syns):
            score = _score_against_chain(syn, chain_s)
            # prefer higher score; break ties by chain size (larger = better)
            if score > best_score or (
                score == best_score and score > 0
                and len(chain_s) > len(chains_syns[best_idx])
            ):
                best_score = score
                best_idx   = idx

        if best_idx >= 0:
            chains_syns [best_idx].append(syn)
            chains_words[best_idx].append((word, syn))
        else:
            chains_syns .append([syn])
            chains_words.append([(word, syn)])

    # ── attach term-frequency and apply size filter ──────────
    result = []
    for chain in chains_words:
        if len(chain) >= MIN_CHAIN_SIZE:
            result.append(
                [(w, s, tf[s.name()]) for (w, s) in chain]
            )

    # Sort chains within document: largest (most coherent) first
    result.sort(key=lambda c: len(c), reverse=True)
    return result


# ─────────────────────────────────────────────────────────────
def build_chains_for_corpus(wsd_docs):
    """
    Apply build_lexical_chains to every document in the corpus.

    Parameters
    ----------
    wsd_docs : list of list of (word_str, Synset | None)

    Returns
    -------
    list of list of list of (word_str, Synset, int)
    """
    chain_docs = []
    total      = len(wsd_docs)

    for i, wsd_pairs in enumerate(wsd_docs):
        if (i + 1) % 100 == 0 or (i + 1) == total:
            print(f"  Building chains: document {i+1}/{total} ...")
        chain_docs.append(build_lexical_chains(wsd_pairs))

    return chain_docs


# ─────────────────────────────────────────────────────────────
def print_chain_stats(chain_docs):
    """Print corpus-level chain statistics."""
    total_docs     = len(chain_docs)
    total_chains   = sum(len(chains) for chains in chain_docs)
    total_concepts = sum(len(c) for chains in chain_docs for c in chains)
    lengths        = [len(c) for chains in chain_docs for c in chains]

    if not lengths:
        print("No chains built.")
        return

    avg_cpd = total_chains   / total_docs     if total_docs    else 0
    avg_cpl = total_concepts / total_chains   if total_chains  else 0
    singles = sum(1 for l in lengths if l == 1)

    print(f"\n{'─'*52}")
    print(f"LEXICAL CHAIN STATISTICS")
    print(f"{'─'*52}")
    print(f"Documents            : {total_docs}")
    print(f"Total chains         : {total_chains}")
    print(f"Avg chains / doc     : {avg_cpd:.1f}")
    print(f"Total concepts       : {total_concepts}")
    print(f"Avg concepts / chain : {avg_cpl:.2f}")
    print(f"Longest chain        : {max(lengths)} concepts")
    print(f"Singleton chains     : {singles}  "
          f"({100 * singles / total_chains:.1f}%)")
    print(f"{'─'*52}")


# ─────────────────────────────────────────────────────────────
def print_document_chains(chain_docs, doc_id,
                           max_chains=10,
                           max_concepts_per_chain=20):
    """Pretty-print the chains for one document."""
    if doc_id >= len(chain_docs):
        print(f"Document {doc_id} does not exist.")
        return

    chains = chain_docs[doc_id]

    print("=" * 80)
    print(f"DOCUMENT {doc_id}  |  {len(chains)} chain(s)")
    print("=" * 80)

    for chain_no, chain in enumerate(chains[:max_chains], start=1):
        print(f"\nCHAIN {chain_no}  [{len(chain)} concept(s)]")
        print("─" * 60)
        for word, syn, freq in chain[:max_concepts_per_chain]:
            defn = syn.definition()
            defn = defn[:50] + "..." if len(defn) > 50 else defn
            print(f"  {word:<15}  {syn.name():<28}  freq={freq}  {defn}")
        if len(chain) > max_concepts_per_chain:
            print(f"  ... and {len(chain) - max_concepts_per_chain} more")


# ─────────────────────────────────────────────────────────────
# ── MAIN EXECUTION ───────────────────────────────────────────
# ─────────────────────────────────────────────────────────────

print("Building lexical chains ...")
print(f"  Hypernym depth   : {HYPERNYM_DEPTH}  (was 3)")
print(f"  WP threshold     : {WP_THRESHOLD}   (was 0.5)")
print(f"  Shared-ancestor  : DISABLED  (was enabled — main explosion source)")
print(f"  Meronym depth    : 1 (direct only)  (was 3)")
print(f"  Algorithm        : linear scan  (was all-pairs Union-Find)")
print()

chain_docs = build_chains_for_corpus(wsd_docs)
print("\nLexical chain construction complete.")
print_chain_stats(chain_docs)

# ── per-document inspection ───────────────────────────────────
print("\nExample — top 5 chains for first 5 documents:\n")
for doc_id in range(min(5, len(chain_docs))):
    print_document_chains(chain_docs, doc_id,
                          max_chains=5,
                          max_concepts_per_chain=10)
    print()

Building lexical chains ...
  Hypernym depth   : 2  (was 3)
  WP threshold     : 0.85   (was 0.5)
  Shared-ancestor  : DISABLED  (was enabled — main explosion source)
  Meronym depth    : 1 (direct only)  (was 3)
  Algorithm        : linear scan  (was all-pairs Union-Find)

  Building chains: document 100/2640 ...
  Building chains: document 200/2640 ...
  Building chains: document 300/2640 ...
  Building chains: document 400/2640 ...
  Building chains: document 500/2640 ...
  Building chains: document 600/2640 ...
  Building chains: document 700/2640 ...
  Building chains: document 800/2640 ...
  Building chains: document 900/2640 ...
  Building chains: document 1000/2640 ...
  Building chains: document 1100/2640 ...
  Building chains: document 1200/2640 ...
  Building chains: document 1300/2640 ...
  Building chains: document 1400/2640 ...
  Building chains: document 1500/2640 ...
  Building chains: document 1600/2640 ...
  Building chains: document 1700/2640 ...
  Building chains: d

In [16]:
# check to make sure chains are right
def print_document_chains(doc_id,
                          max_chains=10,
                          max_concepts_per_chain=20):

    if doc_id >= len(chain_docs):
        print(f"Document {doc_id} does not exist.")
        return

    chains = sorted(
        chain_docs[doc_id],
        key=lambda c: len(c),
        reverse=True
    )

    print("=" * 80)
    print(f"DOCUMENT {doc_id}")
    print(f"Total chains: {len(chains)}")
    print("=" * 80)

    for chain_no, chain in enumerate(chains[:max_chains], start=1):

        print(f"\nCHAIN {chain_no}")
        print(f"Size: {len(chain)} concepts")
        print("-" * 60)

        for stem, syn, weight in chain[:max_concepts_per_chain]:
            print(
                f"{stem:<15}"
                f"{syn.name():<30}"
                f"freq={weight:<3}"
                f"{syn.definition()}"
            )

        if len(chain) > max_concepts_per_chain:
            print(f"... {len(chain)-max_concepts_per_chain} more concepts")



docs_to_check = [0, 1, 2, 3, 4]

for doc_id in range(10):
    print_document_chains(
        doc_id,
        max_chains=5,
        max_concepts_per_chain=15
    )
    print("\n\n")

DOCUMENT 0
Total chains: 54

CHAIN 1
Size: 4 concepts
------------------------------------------------------------
company        company.n.04                  freq=5  organization of performers and associated personnel (especially theatrical)
organization   organization.n.01             freq=4  a group of people who work together
units          unit.n.03                     freq=1  an organization regarded as part of a larger social group
affiliate      affiliate.n.02                freq=2  a subsidiary or subordinate organization that is affiliated with another organization

CHAIN 2
Size: 4 concepts
------------------------------------------------------------
change         change.n.03                   freq=1  the action of changing something
transfer       transfer.n.03                 freq=1  the act of transfering something from one form to another
plays          maneuver.n.03                 freq=1  a deliberate coordinated movement requiring dexterity and skill
shift          s